# GetPythonPath

In [8]:
import sys
print(sys.executable)

c:\Users\peter\anaconda3\python.exe


# Save

In [ ]:
import pandas as pd
import mlflow
import subprocess
import re
import os
from pathlib import Path

########################################################
# Project Folder
########################################################

os.chdir(r"C:\GitTestOnC\Test4_Oct2\Git")

########################################################
# Setup
########################################################

runName = "Test06_Oct2_10.37"
notebookName = "main.ipynb"

mult = 2
add = 4

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(Path(notebookName).stem)

########################################################
# Save Data Version With DVC
########################################################

subprocess.run(["py", "-m", "dvc", "add", "data.csv"], check=True)
subprocess.run(["py", "-m", "dvc", "push"], check=True)

dvcText = Path("data.csv.dvc").read_text()
dvcHash = re.search(r"md5:\s*([0-9a-f]+)", dvcText).group(1)

########################################################
# Save Code + DVC Pointer To Git
########################################################

subprocess.run(["git", "add", notebookName, "data.csv.dvc"], check=True)

hasChanges = subprocess.run(["git", "diff", "--cached", "--quiet"]).returncode != 0

if hasChanges:
    subprocess.run(["git", "commit", "-m", f"Code for {runName}"], check=True)
    subprocess.run(["git", "push"], check=True)

########################################################
# Get Git Commit
########################################################

gitCommit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()

########################################################
# Open Data
########################################################

df = pd.read_csv("data.csv", header=None)

########################################################
# Calc
########################################################

df = add + df * mult
sumResult = df.to_numpy().sum()

########################################################
# Save To MlFlow
########################################################

with mlflow.start_run(run_name=runName):

    mlflow.log_params({"mult": mult, "add": add})
    mlflow.log_metric("sum", sumResult)
    mlflow.set_tag("git_commit", gitCommit)
    mlflow.set_tag("dvc_data_hash", dvcHash)
    mlflow.log_artifact("data.csv.dvc", artifact_path="data")

    ########################################################
    # Export Notebook To HTML
    ########################################################

    Path("results").mkdir(exist_ok=True)

    subprocess.run([r"C:\Users\peter\anaconda3\envs\torchgpu\python.exe", "-m", "jupyter", "nbconvert", "--to", "html", "--output", f"{runName}.html", "--output-dir", "results", notebookName], check=True)

    mlflow.log_artifact(f"results/{runName}.html", artifact_path="notebook")

########################################################
# Save HTML To GitHub
########################################################

subprocess.run(["git", "add", f"results/{runName}.html"], check=True)

hasResultChanges = subprocess.run(["git", "diff", "--cached", "--quiet"]).returncode != 0

if hasResultChanges:
    subprocess.run(["git", "commit", "-m", f"Results for {runName}"], check=True)
    subprocess.run(["git", "push"], check=True)

########################################################
# Tag Experiment
########################################################

subprocess.run(["git", "tag", "-a", runName, "-m", runName], check=True)
subprocess.run(["git", "push", "origin", runName], check=True)

########################################################
# Print
########################################################

print(df)
print("sum =", sumResult)
print("Git commit =", gitCommit)
print("DVC hash =", dvcHash)
print("MlFlow experiment =", Path(notebookName).stem)
print("MlFlow run =", runName)

🏃 View run Test06_Oct2_10.33 at: http://127.0.0.1:5000/#/experiments/2/runs/7f772c79198943aca090b60b903120af
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
    0   1   2
0   6   8  10
1  12  14  16
2  18  20  22
3  24  26  28
sum = 204
Git commit = e2c8deef1727cc3add5c006628468befbd226a3e
DVC hash = 2e53a8406cd074ffd764730c32e3cd00
MlFlow experiment = main
MlFlow run = Test06_Oct2_10.33


In [10]:
df

,0,1,2
0,6,8,10
1,12,14,16
2,18,20,22
3,24,26,28


In [11]:
add, mult

(4, 2)

In [12]:
df = add + df * mult
df.to_numpy().sum()

np.int64(456)